# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [14]:
import pandas as pd

# Table and definitions sample
data = {
    'table': ['SALES', 'CUSTOMERS', 'PRODUCTS'],
    'definition': ['TOTAL SALES', 'CUSTOMER ID', 'PRODUCT TYPE']
}

df = pd.DataFrame(data)
print(df)



       table    definition
0      SALES   TOTAL SALES
1  CUSTOMERS   CUSTOMER ID
2   PRODUCTS  PRODUCT TYPE


In [15]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [16]:
print(text_tables)

SALES: TOTAL SALES
CUSTOMERS: CUSTOMER ID
PRODUCTS: PRODUCT TYPE


In [17]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [21]:
# DEFINITIOSN OF THE QUESTION TO ASK THE MODEL
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="WHAT IS THE TOTAL SALES FOR EACH PRODUCT TYPE?"
)


In [22]:
print(return_OAI(pqt1))

```json
{
    "tables": ["SALES", "PRODUCTS"]
}
```


In [24]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="WHAT IS THE CUSTOMER ID WITH MOST SALES FOR EACH PRODUCT TYPE?")

In [25]:
print(return_OAI(pqt3))

```json
{
    "tables": ["SALES", "CUSTOMERS", "PRODUCTS"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [30]:
import pandas as pd

# Table and definitions sample
data = {
    'table': ['PRODUCTS'],
    'definition': ['SKU, product name, product type, price']
}

df = pd.DataFrame(data)
print(df)


      table                              definition
0  PRODUCTS  SKU, product name, product type, price


In [31]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])
print(text_tables)


PRODUCTS: SKU, product name, product type, price


In [32]:
df_products = pd.DataFrame({
    "SKU": ["ELEC-101", "ELEC-102", "ELEC-103", "ELEC-104", "ELEC-105"],
    "product_name": [
        "Noise-Cancelling Headphones",
        "Smart Home Speaker",
        "4K Action Camera",
        "Wireless Gaming Controller",
        "Portable Power Bank 20000mAh"
    ],
    "product_type": ["ELECTRONICS"] * 5,
    "price": [199.99, 129.00, 249.50, 89.99, 39.90]
})

print(df_products)


        SKU                  product_name product_type   price
0  ELEC-101   Noise-Cancelling Headphones  ELECTRONICS  199.99
1  ELEC-102            Smart Home Speaker  ELECTRONICS  129.00
2  ELEC-103              4K Action Camera  ELECTRONICS  249.50
3  ELEC-104    Wireless Gaming Controller  ELECTRONICS   89.99
4  ELEC-105  Portable Power Bank 20000mAh  ELECTRONICS   39.90


In [38]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Your task:
1. Identify which tables are required to answer the user's SQL question.
2. Based ONLY on the table definitions and the question, determine the final answer.
3. Return everything in a valid JSON format with this structure:

{{
  "tables": [...],
  "answer": "..."
}}

###User Question:
{question}
"""



In [39]:
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="WHAT IS THE MOST EXPENSIVE PRODUCT?"
)

print(return_OAI(pqt1))


{
  "tables": [
    "PRODUCTS"
  ],
  "answer": "The most expensive product is ..."
}


In [40]:
pqt2 = prompt_question_tables.format(
    tables=text_tables,
    question="WHAT IS THE CHEAPEST PRODUCT?"
)

print(return_OAI(pqt2))


{
  "tables": [
    "PRODUCTS"
  ],
  "answer": "The cheapest product is ..."
}


## Conclusion Summary

In this exercise, we explored how a language model can identify which database tables are needed to answer a user’s SQL question based solely on table names and short content definitions. We recreated the workflow from the original lab, built a new PRODUCTS table with realistic electronic items, and tested prompts designed to make the model return the relevant tables. We also attempted to extend the output so the model would provide the final answer (e.g., the most expensive or cheapest product).

A key learning was that the model can correctly determine which tables are required, as long as the prompt is clear and the table definitions are simple. It consistently returned "PRODUCTS" for both questions, which matches the expected behavior. However, we also discovered an important limitation: the model cannot compute actual results (such as identifying the most expensive product) unless the underlying data is explicitly provided in the prompt. When only table definitions are available, the model can only return placeholders like "The most expensive product is ...". This behavior is expected, because the model has no access to the real dataset unless we include it.

We also learned that prompt formatting matters. When using Python’s .format() method, JSON braces must be escaped ({{ }}), otherwise Python interprets them as formatting placeholders and raises errors. This reinforced the importance of careful prompt construction when mixing code and natural language instructions.

Overall, the exercise demonstrated that language models are effective at schema‑level reasoning (selecting tables) but cannot infer data‑level answers without being given the actual rows. It also highlighted the value of prompt engineering, especially when working with structured outputs like JSON.